# Entrainement LightGCN sur Colab (GPU)

Notebook fin : tout le code vit dans le repo (`src/`, `scripts/`), ce notebook ne fait que cloner, installer, et appeler les memes scripts que ceux utilises en local -- aucune duplication de logique, aucune adaptation entre local (CPU) et Colab (GPU), le device est detecte automatiquement dans `scripts/train_lightgcn.py`.

**Pre-requis** : la branche `feature/models-lightgcn` doit avoir ete poussee sur GitHub (`git push origin feature/models-lightgcn`) avant d'executer ce notebook.

**Pourquoi Colab** : l'etude d'ablation entraine LightGCN pour 5 profondeurs successives (K=1..5) ; un GPU accelere nettement les 5 entrainements par rapport au CPU local.

In [ ]:
# Verifie qu'un GPU est bien attribue (Runtime > Change runtime type > GPU)
import torch
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = "https://github.com/Prosper015/recommender-gnn-vs-cf.git"
BRANCH = "feature/models-lightgcn"

!git clone --branch "$BRANCH" "$GITHUB_REPO" repo
%cd repo

In [ ]:
# Colab fournit deja torch (avec CUDA) ; on evite de le reinstaller pour ne
# pas perdre le build CUDA preconfigure. On installe uniquement le reste.
!pip install -q pandas requests tqdm pyarrow mlflow scikit-surprise matplotlib pytest tabulate

In [ ]:
# Sanity check rapide avant de lancer les entrainements longs
!python -m pytest tests/ -q

In [ ]:
# Etude d'ablation complete (K=1..5) -- produit results/ablation_depth.csv,
# results/ablation_depth.png, et sauvegarde le meilleur modele dans models/
!python -m scripts.run_ablation --dataset 100k --epochs 50

In [ ]:
# Baselines (peuvent aussi tourner en local, mais autant tout faire ici en une fois)
!python -m scripts.train_baselines --dataset 100k
!python -m scripts.build_comparison_table

## Recuperer les resultats

Les fichiers produits (`models/*.pt`, `models/*.pkl`, `models/id_mappings.json`, `results/*.csv`, `results/*.png`) sont dans le systeme de fichiers ephemere de Colab. Telecharge-les avant de fermer la session, ou monte Google Drive en amont si tu veux qu'ils persistent.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("lightgcn_artifacts", "zip", ".", "models")
shutil.make_archive("results_artifacts", "zip", ".", "results")
files.download("lightgcn_artifacts.zip")
files.download("results_artifacts.zip")